In [1]:
import os

for dirname ,_ ,filenames in os.walk("/kaggle/input"):
    if filenames:
        print(dirname,"->",len(filenames),"files")

/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX -> 1 files
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray -> 3 files
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/test -> 3 files
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/test/NORMAL -> 234 files
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/test/PNEUMONIA -> 390 files
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/train -> 3 files
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/train/NORMAL -> 1342 files
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/train/PNEUMONIA -> 3876 files
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/val -> 1 

nspect image filenames to extract potential grouping identifiers. These identifiers can help investigate whether related images may be distributed across training, validation, and test sets. Filename-derived groups are only proxies and must be verified before they can be treated as patient identifiers.

In [2]:
import re
import pandas as pd
from pathlib import Path

records = []
DATA_DIR = Path(
    "/kaggle/input/datasets/paultimothymooney/"
    "chest-xray-pneumonia/chest_xray/chest_xray"
)

for label in ["NORMAL", "PNEUMONIA"]:
    folder = DATA_DIR / "train" / label

    for path in folder.iterdir():
        if path.is_file() and path.suffix.lower() in [
            ".jpg", ".jpeg", ".png"
        ]:
            records.append({
                "path": str(path),
                "filename": path.name,
                "label": label
            })

check_df = pd.DataFrame(records)

def extract_group(filename, label):
    if label == "PNEUMONIA":
        match = re.match(r"(person\d+)_", filename)
        return match.group(1) if match else None

    if label == "NORMAL":
        match = re.match(r"(?:(NORMAL2-)?)(IM-\d+)", filename)
        return (
            f"{match.group(1) or ''}{match.group(2)}"
            if match else None
        )

    return None

check_df["group"] = check_df.apply(
    lambda row: extract_group(row["filename"], row["label"]),
    axis=1
)

print("Total images:", len(check_df))
print("Missing group IDs:", check_df["group"].isna().sum())

Total images: 5216
Missing group IDs: 0


Split the training images into training and validation subsets using filename-derived groups. This aims to reduce overlap between related images across the two subsets. Because these groups are inferred from filenames rather than verified patient identifiers, the split should not be described as confirmed patient-level separation.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_parts = []
val_parts = []

for label in ["NORMAL", "PNEUMONIA"]:
    subset = check_df[check_df["label"] == label].copy()

    train_idx, val_idx = next(
        splitter.split(subset, groups=subset["group"])
    )

    train_parts.append(subset.iloc[train_idx])
    val_parts.append(subset.iloc[val_idx])

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)

train_groups = set(train_df["group"])
val_groups = set(val_df["group"])

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

print("\nTraining class counts:")
print(train_df["label"].value_counts())

print("\nValidation class counts:")
print(val_df["label"].value_counts())

print("\nOverlapping groups:", len(train_groups & val_groups))


Training images: 4158
Validation images: 1058

Training class counts:
label
PNEUMONIA    3087
NORMAL       1071
Name: count, dtype: int64

Validation class counts:
label
PNEUMONIA    788
NORMAL       270
Name: count, dtype: int64

Overlapping groups: 0


In [4]:

train_counts = train_df["label"].value_counts()
total = train_counts.sum()

print(f"Total training images: {total}\n")

for label, count in train_counts.items():
    percentage = count / total * 100
    print(f"{label}: {count} images ({percentage:.2f}%)")


Total training images: 4158

PNEUMONIA: 3087 images (74.24%)
NORMAL: 1071 images (25.76%)


In [5]:

import torch

counts = train_df["label"].value_counts()

normal_count = counts["NORMAL"]
pneumonia_count = counts["PNEUMONIA"]
total = normal_count + pneumonia_count

class_weights = torch.tensor(
    [
        total / (2 * normal_count),
        total / (2 * pneumonia_count)
    ],
    dtype=torch.float32
)

print("NORMAL weight:", class_weights[0].item())
print("PNEUMONIA weight:", class_weights[1].item())


NORMAL weight: 1.941176414489746
PNEUMONIA weight: 0.6734693646430969


Baseline Model: ResNet18 Transfer Learning

Establish a baseline using a pretrained ResNet18 model adapted for binary classification of chest X-ray images into NORMAL and PNEUMONIA classes. Class-weighted cross-entropy loss is used to address class imbalance. The baseline provides a reference point for evaluating 
the effect of data augmentation in the improved model.


In [6]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from torchvision import transforms,models

class mydata(Dataset):
    def __init__(self,frame):
        self.x=frame
        self.transform=transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
    def __len__(self):
        return len(self.x)
        
    def __getitem__(self,idx):
        row = self.x.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        image = self.transform(image)
        label = 0 if row["label"] == "NORMAL" else 1
        return image, torch.tensor(label, dtype=torch.long)

def training_loop(model,train_loader,optimizer,loss_fn,device):
    model = model.to(device)
    model.train()
    total_loss=0
    correct=0
    total=0
    for img,label in train_loader:
            
        img = img.to(device)
        label = label.to(device)
            
        optimizer.zero_grad()
        pred = model(img)
        loss =  loss_fn(pred,label)

        loss.backward()
        optimizer.step()
            
        total_loss += loss.item()
        predicted = pred.argmax(dim=1)
        correct += (predicted == label).sum().item()
        total += label.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total


            
    print(
        f"epoch:{epoch+1}|"
        f"the avarage loss is:{avg_loss:.4f}|"
        f"the accuracy is:{accuracy:.4f}|" )
    return model

def val_loop(model,val_loader,loss_fn, device):
    model = model.to(device)
    model.eval()

    all_labels = []
    all_probs = []
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
         for img,label in val_loader:
                
            img = img.to(device)
            label = label.to(device)
        
            pred = model(img)
            
            probs = torch.softmax(pred, dim=1)
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(label.cpu().numpy())
            
            loss = loss_fn(pred, label)
        
            total_loss += loss.item() * label.size(0)
        
            predicted = pred.argmax(dim=1)
            correct += (predicted == label).sum().item()
            total += label.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    auc = roc_auc_score(all_labels, all_probs)
    all_preds = [1 if p >= 0.5 else 0 for p in all_probs]
    
    print(classification_report(
        all_labels,
        all_preds,
        labels=[0, 1],
        target_names=["NORMAL", "PNEUMONIA"],
        zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(all_labels, all_preds))

    print(
        f"Validation Loss: {avg_loss:.4f} | "
        f"Validation Accuracy: {accuracy:.4f}|"
        f"ROC-AUC: {auc:.4f}"
    )
    
best_checkpoint_path = "/kaggle/working/pneumonia_resnet18_epoch2.pth"
train_data=mydata(train_df)
val_data=mydata(val_df)

train_loader = DataLoader(train_data,batch_size=32,shuffle=True)
val_loader = DataLoader(val_data,batch_size=32,shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights="DEFAULT")
model.fc = nn.Linear(model.fc.in_features,2)
optimizer = optim.Adam(model.parameters(),lr=1e-4)
loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(device))



for epoch in range(3):
    training_loop(model, train_loader, optimizer, loss_fn, device)

    val_loop(model, val_loader, loss_fn, device)
    if epoch == 1:
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "class_to_idx": {"NORMAL": 0, "PNEUMONIA": 1},
            "image_size": 224
        }, best_checkpoint_path)

        print("Epoch 2 checkpoint saved!")


epoch:1|the avarage loss is:0.1136|the accuracy is:0.9555|
              precision    recall  f1-score   support

      NORMAL       0.83      0.99      0.91       270
   PNEUMONIA       1.00      0.93      0.96       788

    accuracy                           0.95      1058
   macro avg       0.91      0.96      0.93      1058
weighted avg       0.96      0.95      0.95      1058

Confusion matrix:
[[268   2]
 [ 54 734]]
Validation Loss: 0.1666 | Validation Accuracy: 0.9471|ROC-AUC: 0.9947
              precision    recall  f1-score   support

      NORMAL       0.95      0.99      0.97       270
   PNEUMONIA       1.00      0.98      0.99       788

    accuracy                           0.98      1058
   macro avg       0.97      0.99      0.98      1058
weighted avg       0.99      0.98      0.98      1058

Confusion matrix:
[[268   2]
 [ 14 774]]
Validation Loss: 0.0550 | Validation Accuracy: 0.9849|ROC-AUC: 0.9989
Epoch 2 checkpoint saved!
epoch:3|the avarage loss is:0.0055|the 

In [8]:
from pathlib import Path

print(Path(best_checkpoint_path).exists())

True


In [9]:

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


In [10]:
import torch
checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Loaded checkpoint from epoch:", checkpoint["epoch"])


Loaded checkpoint from epoch: 2


In [11]:
from pathlib import Path
import pandas as pd

test_records = []

for label in ["NORMAL", "PNEUMONIA"]:
    folder = DATA_DIR / "test" / label

    for path in folder.iterdir():
        if path.is_file() and path.suffix.lower() in [
            ".jpg", ".jpeg", ".png"
        ]:
            test_records.append({
                "path": str(path),
                "label": label
            })

test_df = pd.DataFrame(test_records)

print("Test images:", len(test_df))
print("\nClass counts:")
print(test_df["label"].value_counts())

Test images: 624

Class counts:
label
PNEUMONIA    390
NORMAL       234
Name: count, dtype: int64


### Baseline Model Evaluation on the Held-Out Test Set

Evaluate the baseline ResNet18 model using the held-out test set. Report per-class precision, recall, and F1-score, along with the confusion matrix and ROC-AUC, to assess classification performance beyond accuracy alone.

In [12]:

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)
import torch

test_data = mydata(test_df)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
model.eval()

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

print("TEST CLASSIFICATION REPORT")
print(classification_report(
    all_labels,
    all_preds,
    labels=[0, 1],
    target_names=["NORMAL", "PNEUMONIA"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(all_labels, all_preds, labels=[0, 1]))

print("Test ROC-AUC:", round(roc_auc_score(all_labels, all_probs), 4))


TEST CLASSIFICATION REPORT
              precision    recall  f1-score   support

      NORMAL       0.99      0.63      0.77       234
   PNEUMONIA       0.82      1.00      0.90       390

    accuracy                           0.86       624
   macro avg       0.91      0.81      0.84       624
weighted avg       0.88      0.86      0.85       624

Confusion matrix:
[[148  86]
 [  1 389]]
Test ROC-AUC: 0.9618


# Improved Model: ResNet18 with Data Augmentation

This experiment improves the baseline ResNet18 classifier by applying mild rotation and brightness/contrast augmentation to training images. Weighted cross-entropy loss addresses class imbalance. The model with the highest validation ROC-AUC is saved and subsequently evaluated on the held-out test set.

Augmentation is applied only during training; validation and test images use the same deterministic preprocessing.

In [14]:
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from PIL import Image
from torchvision import transforms, models

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    accuracy_score
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class mydata(Dataset):
    def __init__(self, frame, transform=None):
        self.x = frame.reset_index(drop=True).copy()
        self.transform = transform if transform is not None else eval_transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        row = self.x.iloc[idx]

        with Image.open(row["path"]) as img:
            image = img.convert("RGB")

        image = self.transform(image)

        label = 0 if row["label"] == "NORMAL" else 1

        return image, torch.tensor(label, dtype=torch.long)


train_data = mydata(train_df, transform=train_transform)
val_data = mydata(val_df, transform=eval_transform)
test_data = mydata(test_df, transform=eval_transform)

train_loader = DataLoader(
    train_data, batch_size=32, shuffle=True,
    num_workers=2, pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_data, batch_size=32, shuffle=False,
    num_workers=2, pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_data, batch_size=32, shuffle=False,
    num_workers=2, pin_memory=(device.type == "cuda")
)

model = models.resnet18(weights="DEFAULT")
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(device))


def training_loop(model, loader, optimizer, loss_fn, device, epoch):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    print(
        f"Epoch {epoch}: "
        f"Train loss={total_loss / total:.4f} | "
        f"Train accuracy={correct / total:.4f}"
    )


def evaluate(model, loader, loss_fn, device, split_name):
    model.eval()

    labels_all = []
    probs_all = []
    total_loss = 0.0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels_device = labels.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels_device)
            probs = torch.softmax(outputs, dim=1)[:, 1]

            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)

            labels_all.extend(labels.numpy())
            probs_all.extend(probs.cpu().numpy())

    y_true = np.array(labels_all)
    y_probs = np.array(probs_all)
    y_pred = (y_probs >= 0.5).astype(int)

    auc = roc_auc_score(y_true, y_probs)
    accuracy = accuracy_score(y_true, y_pred)

    print(f"\n--- {split_name} ---")
    print(f"Loss: {total_loss / total:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"ROC-AUC: {auc:.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["NORMAL", "PNEUMONIA"],
        digits=3,
        zero_division=0
    ))

    return auc

best_auc = -1.0
best_checkpoint_path = "/kaggle/working/pneumonia_resnet18_augmented_best.pth"

for epoch in range(1, 4):
    training_loop(
        model, train_loader, optimizer, loss_fn, device, epoch
    )

    val_auc = evaluate(
        model, val_loader, loss_fn, device,
        f"VALIDATION — Epoch {epoch}"
    )

    if val_auc > best_auc:
        best_auc = val_auc

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_auc": best_auc,
            "class_to_idx": {"NORMAL": 0, "PNEUMONIA": 1},
            "image_size": 224
        }, best_checkpoint_path)

        print(f"Saved best checkpoint: epoch {epoch}, AUC={best_auc:.4f}")

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("\nBest epoch:", checkpoint["epoch"])
print("Best validation AUC:", round(checkpoint["best_val_auc"], 4))

evaluate(
    model, test_loader, loss_fn, device,
    "FINAL TEST — Augmented model"
)

Device: cuda
Epoch 1: Train loss=0.1155 | Train accuracy=0.9533

--- VALIDATION — Epoch 1 ---
Loss: 0.0553
Accuracy: 0.9839
ROC-AUC: 0.9974
Confusion matrix:
[[262   8]
 [  9 779]]
              precision    recall  f1-score   support

      NORMAL      0.967     0.970     0.969       270
   PNEUMONIA      0.990     0.989     0.989       788

    accuracy                          0.984      1058
   macro avg      0.978     0.979     0.979      1058
weighted avg      0.984     0.984     0.984      1058

Saved best checkpoint: epoch 1, AUC=0.9974
Epoch 2: Train loss=0.0527 | Train accuracy=0.9808

--- VALIDATION — Epoch 2 ---
Loss: 0.0708
Accuracy: 0.9773
ROC-AUC: 0.9981
Confusion matrix:
[[265   5]
 [ 19 769]]
              precision    recall  f1-score   support

      NORMAL      0.933     0.981     0.957       270
   PNEUMONIA      0.994     0.976     0.985       788

    accuracy                          0.977      1058
   macro avg      0.963     0.979     0.971      1058
weighted 

np.float64(0.9759040105193951)

In [16]:

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Loaded best checkpoint from epoch:", checkpoint["epoch"])
print("Best validation AUC:", checkpoint["best_val_auc"])

evaluate(
    model,
    test_loader,
    loss_fn,
    device,
    "FINAL TEST — Augmented model"
)


Loaded best checkpoint from epoch: 3
Best validation AUC: 0.9991586764429403

--- FINAL TEST — Augmented model ---
Loss: 0.4764
Accuracy: 0.8830
ROC-AUC: 0.9759
Confusion matrix:
[[162  72]
 [  1 389]]
              precision    recall  f1-score   support

      NORMAL      0.994     0.692     0.816       234
   PNEUMONIA      0.844     0.997     0.914       390

    accuracy                          0.883       624
   macro avg      0.919     0.845     0.865       624
weighted avg      0.900     0.883     0.877       624



np.float64(0.9759040105193951)

## Discussion: Validation–Test Performance Gap

The augmented ResNet18 model achieved a validation ROC-AUC of approximately **0.9992**, but its ROC-AUC on the held-out test set was **0.9759**. A similar difference was observed in accuracy: validation accuracy reached approximately **98.7%**, while test accuracy was **88.3%**.

This difference indicates that the model performed better on the validation data than on the unseen test data. Although the test ROC-AUC remains high, the gap suggests that the model's generalization performance requires further investigation.

Several factors could contribute to this difference, including differences in the distributions of the training, validation, and test images, overfitting to patterns in the training data, or similarities between images within the dataset partitions. The current experiment does not establish which factor is responsible, so further analysis is needed before drawing a definitive conclusion.

The confusion matrix also reveals an important limitation. The model correctly classified 162 out of 234 NORMAL images, while incorrectly classifying 72 as PNEUMONIA. In contrast, it correctly classified 389 out of 390 PNEUMONIA images. This indicates that the model has very high pneumonia recall but a substantially lower recall for normal images at the current classification threshold of 0.5.

### Conclusion

These results demonstrate that the augmented model can distinguish between the two classes effectively in terms of ROC-AUC, but its classification behavior is not equally reliable for both classes. Future work should investigate the validation–test performance gap, examine misclassified images, verify the independence of the data partitions where possible, and evaluate alternative classification thresholds using validation data only.

This experiment is an educational machine-learning project and should not be interpreted as evidence of clinical diagnostic readiness.